# Qwen3.5-0.8B next-edit LoRA smoke (tinycomplete)
Human-run Colab notebook. Preferred GPU: **L4 24GB** (bf16). Fallback: T4 16GB (fp16 path, watch GDN NaNs). Short cells call repo code in `src/tinycomplete/train/train.py`.

In [ ]:
# Cell 1 — configuration
MODEL_ID = "Qwen/Qwen3.5-0.8B-Base"
RUN_LORA_SMOKE = True
RUN_FULL_SMOKE = False
MAX_SEQ_LENGTH = 2048
MAX_STEPS = 100
SEED = 42
COLAB_CU_PER_HOUR = None  # fill in from the Colab pay-as-you-go panel, e.g. 0.0
print(MODEL_ID, MAX_SEQ_LENGTH, MAX_STEPS, SEED)

In [ ]:
# Cell 2 — runtime check (CUDA required)
import torch
assert torch.cuda.is_available(), "This notebook requires a CUDA GPU runtime."
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 2**30, "GiB")

In [ ]:
# Cell 2b — driver check
!nvidia-smi

In [ ]:
# Cell 3 — repository + dependencies (do NOT replace Colab CUDA torch)
# First run: clone the public repo, then install. Re-runs: skip clone.
!test -d tabcomplete || git clone https://github.com/anomalyco/tabcomplete.git
%cd tabcomplete
!pip install -q unsloth  # pulls trl/peft/datasets; keeps Colab's CUDA torch
!pip install -q pyyaml pydantic tree-sitter tree-sitter-language-pack unidiff
import unsloth, transformers, torch, trl
print("unsloth", unsloth.__version__)
print("transformers", transformers.__version__)  # Qwen3.5 needs transformers v5+
print("torch", torch.__version__, torch.version.cuda)
print("trl", trl.__version__)

In [ ]:
# Cell 4 — dataset (tiny fixture set; asserts records exist)
import json
from collections import Counter
from tinycomplete.data.fim import generate_examples
from tinycomplete.data.static_edits import generate_next_edits
from tinycomplete.train.train import tokenize_records
from transformers import AutoTokenizer

SRC = open("tests/fixtures/sample.py").read()
examples = generate_examples(SRC, 40, SEED, source_path="tests/fixtures/sample.py")
examples += generate_next_edits(SRC, 40, SEED + 1, source_path="tests/fixtures/sample.py")
records = [e.model_dump() for e in examples]
assert records, "no records generated"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
ds, stats = tokenize_records(records, tok, MAX_SEQ_LENGTH)
print("examples:", stats["n"] if isinstance(stats, dict) else len(ds))
print("median/p95/max tokens:", stats["median"], stats["p95"], stats["max"])
print("provenance:", dict(Counter(e["provenance"] for e in records)))
print("modes:", dict(Counter(e["mode"] for e in records)))
with open("/tmp/train.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")
print("wrote /tmp/train.jsonl")

In [ ]:
# Cell 5 — LoRA smoke training (first GPU experiment)
from tinycomplete.train.train import TrainConfig, train

if RUN_LORA_SMOKE:
    cfg = TrainConfig(model_id=MODEL_ID, mode="lora", max_seq_length=MAX_SEQ_LENGTH,
                      max_steps=MAX_STEPS, seed=SEED, dataset_path="/tmp/train.jsonl")
    lora_report = train(cfg)
    print(lora_report)
if RUN_FULL_SMOKE:
    cfg = TrainConfig(model_id=MODEL_ID, mode="full", max_seq_length=1024,
                      max_steps=15, seed=SEED, dataset_path="/tmp/train.jsonl")
    full_report = train(cfg)
    print(full_report)

In [ ]:
# Cell 6 — save checkpoint (Drive optional; /content is ephemeral)
import os, shutil
try:
    from google.colab import drive
    drive.mount("/content/drive", timeout_ms=60000)
    DEST = "/content/drive/MyDrive/tinycomplete/checkpoints"
except Exception as e:
    print("Drive unavailable, checkpoints stay in /content:", e)
    DEST = "/content/tinycomplete-checkpoints"
os.makedirs(DEST, exist_ok=True)
shutil.copy("outputs/qwen35-lora-smoke/throughput.json", DEST)
print("saved throughput.json to", DEST)

In [ ]:
# Cell 7 — reload + eval (fixed tiny validation set, before/after loss)
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycomplete.train.train import load_jsonl_records, tokenize_records

tok = AutoTokenizer.from_pretrained(MODEL_ID)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
ft = PeftModel.from_pretrained(base, "outputs/qwen35-lora-smoke")
recs = load_jsonl_records("/tmp/train.jsonl")[-8:]
val, _ = tokenize_records(recs, tok, MAX_SEQ_LENGTH)
base.eval(); ft.eval()
import torch.nn.functional as F
def nll(model, ids):
    t = torch.tensor([ids], device=model.device)
    with torch.no_grad():
        out = model(t)
    return F.cross_entropy(out.logits[0, :-1], t[0, 1:]).item()
for i, r in enumerate(val[:3]):
    b, a = nll(base, r["input_ids"]), nll(ft, r["input_ids"])
    print(f"ex{i}: base NLL={b:.3f} lora NLL={a:.3f}")
prompt = tok.encode("def add(a, b):", return_tensors="pt").to(ft.device)
print(tok.decode(ft.generate(prompt, max_new_tokens=24, do_sample=False)[0]))

In [ ]:
# Cell 8 — throughput report (T4 vs L4 vs A100 comparison metric)
r = dict(lora_report)
eff_batch = 2 * 4  # per_device_batch_size x gradient_accumulation (see LoRA smoke config)
r["effective_batch_size"] = eff_batch
r["sequence_length"] = MAX_SEQ_LENGTH
if COLAB_CU_PER_HOUR:
    cu = (r["wall_time_s"] / 3600) * COLAB_CU_PER_HOUR
    r["compute_units_used"] = cu
    r["training_tokens_per_compute_unit"] = r["train_tokens"] / cu
print(r)